# 06 — Fire susceptibility
## Where is the Galičica landscape relatively more susceptible to burning?

In Practical 05 we separated **fire weather** from **fuel/vegetation condition**. Here we build a deliberately simple and interpretable **landscape susceptibility** model.

The question is:

> **If an ignition occurs under fire-conducive weather, which parts of the landscape have relatively more favorable terrain and vegetation conditions for fire?**

We combine four factors:

- broad vegetation/fuel type;
- typical summer vegetation dryness;
- slope;
- aspect-derived southness.

The model is a **weighted overlay**, not a machine-learning black box.

> **Important:** this is a **relative landscape susceptibility map**, not an ignition-probability map, not an operational fire-danger product, and not societal risk. Weather, ignition sources, suppression capacity, people and assets are not included.

### How to work with this notebook

- Run the **core** cells from top to bottom.
- Pause at the interpretation questions before moving on.
- Most code is provided; the task is to understand the workflow, change selected parameters, and defend the interpretation.
- If a live service fails, tell a trainer rather than spending the practical debugging infrastructure.
- Stretch tasks are optional and are intended for participants who finish the core workflow early.

## 1. Imports and Earth Engine

In [ ]:
from pathlib import Path
import os

import ee
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
import matplotlib.pyplot as plt

GEE_PROJECT_ID = os.environ.get("GEE_PROJECT_ID", "").strip()

def initialize_earth_engine():
    """Initialize Earth Engine using existing credentials or the normal auth flow."""
    try:
        if GEE_PROJECT_ID:
            ee.Initialize(project=GEE_PROJECT_ID)
        else:
            ee.Initialize()
    except Exception:
        print("Earth Engine authentication is required.")
        ee.Authenticate()
        try:
            if GEE_PROJECT_ID:
                ee.Initialize(project=GEE_PROJECT_ID)
            else:
                ee.Initialize()
        except Exception as exc:
            raise RuntimeError(
                "Earth Engine could not initialize. If your account requires a "
                "registered Google Cloud project, set the GEE_PROJECT_ID "
                "environment variable, restart the kernel, and run this cell again."
            ) from exc

initialize_earth_engine()
print("Earth Engine ready.")

## 2. Load the canonical AOI and EFFIS fire-history layer

We use the same course inputs as in the previous practicals:

- `data/aoi/galicica_aoi.geojson`
- `data/effis/Galicica.gpkg`

The EFFIS polygons are used only as an **independent plausibility check** later. They are not used to fit the weighted-overlay model.

In [ ]:
repo_root = Path.home() / "mystorage" / "fire-school"

def find_course_file(relative_path):
    candidates = [
        repo_root / relative_path,
        Path.cwd() / relative_path,
        Path.cwd().parent / relative_path,
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(
            f"{relative_path} was not found. Run git pull in the fire-school repository."
        )
    return path

AOI_PATH = find_course_file(Path("data/aoi/galicica_aoi.geojson"))
EFFIS_PATH = find_course_file(Path("data/effis/Galicica.gpkg"))

aoi_gdf = gpd.read_file(AOI_PATH).to_crs("EPSG:4326")
effis = gpd.read_file(EFFIS_PATH).to_crs("EPSG:4326").copy()

aoi_geom = aoi_gdf.geometry.iloc[0]
AOI = ee.Geometry(aoi_geom.__geo_interface__)

effis["FIREDATE"] = pd.to_datetime(effis["FIREDATE"], errors="coerce")
if "AREA_HA" in effis.columns:
    effis["AREA_HA"] = pd.to_numeric(effis["AREA_HA"], errors="coerce")

effis_features = [
    ee.Feature(ee.Geometry(geom.__geo_interface__))
    for geom in effis.geometry
    if geom is not None and not geom.is_empty
]
EFFIS_FC = ee.FeatureCollection(effis_features)
EFFIS_GEOM = EFFIS_FC.geometry()

print("AOI area (km²):", round(AOI.area().divide(1e6).getInfo(), 1))
print("EFFIS polygons:", len(effis_features))
print(
    "EFFIS years:",
    int(effis["FIREDATE"].dt.year.min()),
    "→",
    int(effis["FIREDATE"].dt.year.max()),
)

## 3. What exactly are we modelling?

A useful terminology hierarchy for this course is:

- **susceptibility** — relatively persistent landscape conditions that favor burning/spread;
- **fire weather / danger** — dynamic meteorological conditions that change daily or hourly;
- **occurrence** — whether a fire actually happened;
- **risk** — potential consequences for people, ecosystems, infrastructure or other valued assets.

This notebook focuses on the first item.

We deliberately **exclude 2024 event weather** from the susceptibility score. Otherwise we would mix a relatively static landscape model with a particular event's dynamic conditions.

## 4. Factor 1 — broad vegetation / fuel-type proxy

ESA WorldCover is not a fuel model, but broad vegetation classes provide a transparent first approximation of where burnable vegetation is present.

For this teaching exercise we assign relative scores from 0 to 1:

| WorldCover class | Score |
|---|---:|
| Shrubland | 1.00 |
| Grassland | 0.90 |
| Tree cover | 0.85 |
| Cropland | 0.65 |
| Moss / lichen | 0.30 |
| Wetland | 0.20 |
| Bare / sparse vegetation | 0.15 |
| Built-up / water / snow-ice | 0.00 |

These values are **illustrative expert weights**, not universal physical constants.

In [ ]:
worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
    .clip(AOI)
)

WC_CLASSES = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
FUEL_SCORES = [0.85, 1.00, 0.90, 0.65, 0.00, 0.15, 0.00, 0.00, 0.20, 0.50, 0.30]

fuel_score = (
    worldcover
    .remap(WC_CLASSES, FUEL_SCORES, 0)
    .rename("fuel_score")
)

# Mask obviously non-burnable classes from the final product.
burnable_mask = (
    worldcover.neq(50)   # built-up
    .And(worldcover.neq(70))  # snow / ice
    .And(worldcover.neq(80))  # permanent water
)

## 5. Factor 2 — typical summer vegetation dryness

We use Sentinel-2 NDMI as a **live-vegetation moisture proxy**.

To avoid building the map around the 2024 fire itself, we use a multi-year pre-fire reference:

**July–August, 2019–2023**

For every year we create a cloud-masked seasonal median, calculate NDMI, and then take the median across years.

Lower typical summer NDMI receives a higher relative dryness score.

> NDMI does **not** measure dead-fuel moisture or fuel load.

In [ ]:
MAX_CLOUD = 60

def mask_s2_scl(img):
    scl = img.select("SCL")
    bad = (
        scl.eq(3)
        .Or(scl.eq(8))
        .Or(scl.eq(9))
        .Or(scl.eq(10))
        .Or(scl.eq(11))
    )
    return (
        img.updateMask(bad.Not())
        .select(["B8", "B11"])
    )

def summer_ndmi(year):
    start = f"{year}-07-01"
    end = f"{year}-09-01"

    col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(AOI)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD))
        .map(mask_s2_scl)
    )

    composite = col.median()
    return (
        composite
        .normalizedDifference(["B8", "B11"])
        .rename("NDMI")
        .set("year", year)
        .set("n_scenes", col.size())
    )

ndmi_years = [summer_ndmi(y) for y in range(2019, 2024)]
typical_summer_ndmi = (
    ee.ImageCollection.fromImages(ndmi_years)
    .median()
    .clip(AOI)
    .rename("typical_summer_NDMI")
)

scene_counts = {
    y: int(ee.Image(img).get("n_scenes").getInfo())
    for y, img in zip(range(2019, 2024), ndmi_years)
}
scene_counts

### Convert NDMI to a relative dryness score

Instead of imposing a universal NDMI threshold, we scale the Galičica distribution using its 10th and 90th percentiles:

- low NDMI → score near 1;
- high NDMI → score near 0.

This makes the factor **relative to the study area**.

In [ ]:
ndmi_pct = (
    typical_summer_ndmi
    .reduceRegion(
        reducer=ee.Reducer.percentile([10, 90]),
        geometry=AOI,
        scale=20,
        maxPixels=1e9,
        bestEffort=True,
        tileScale=2,
    )
    .getInfo()
)

NDMI_P10 = float(ndmi_pct["typical_summer_NDMI_p10"])
NDMI_P90 = float(ndmi_pct["typical_summer_NDMI_p90"])

if NDMI_P90 <= NDMI_P10:
    raise RuntimeError("Unexpected NDMI percentile range.")

dryness_score = (
    ee.Image(1)
    .subtract(
        typical_summer_ndmi
        .subtract(NDMI_P10)
        .divide(NDMI_P90 - NDMI_P10)
        .clamp(0, 1)
    )
    .rename("dryness_score")
)

print("Typical summer NDMI p10:", round(NDMI_P10, 3))
print("Typical summer NDMI p90:", round(NDMI_P90, 3))

## 6. Factors 3 and 4 — slope and southness

Terrain affects fire behavior and local drying.

We derive terrain from SRTM:

- **slope score:** increases linearly from 0 at 0° to 1 at 30° and remains capped thereafter;
- **southness:** 1 for due south, 0 for due north, with intermediate values in between.

Southness is relevant here because the study area is in the Northern Hemisphere. It is a proxy for potential solar exposure, not direct measured fuel moisture.

In [ ]:
dem = ee.Image("USGS/SRTMGL1_003").select("elevation").clip(AOI)
terrain = ee.Terrain.products(dem)

slope = terrain.select("slope").rename("slope_deg")
aspect = terrain.select("aspect").rename("aspect_deg")

slope_score = (
    slope
    .divide(30)
    .clamp(0, 1)
    .rename("slope_score")
)

southness = (
    aspect
    .subtract(180)
    .multiply(np.pi / 180)
    .cos()
    .add(1)
    .divide(2)
    .rename("southness_score")
)

print("Terrain factors ready.")

## 7. Inspect the four factor layers

Before combining them, inspect them separately. A weighted overlay is only as defensible as its input layers.

In [ ]:
def add_ee_layer(m, ee_image, vis_params, name):
    map_id = ee_image.getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id["tile_fetcher"].url_format,
        attr="Google Earth Engine",
        name=name,
        overlay=True,
        control=True,
    ).add_to(m)

centroid = aoi_geom.centroid
CENTER = [centroid.y, centroid.x]

m = folium.Map(location=CENTER, zoom_start=10)

folium.GeoJson(
    aoi_gdf,
    name="Course AOI",
    style_function=lambda _: {
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.0,
    },
).add_to(m)

add_ee_layer(
    m,
    fuel_score,
    {"min": 0, "max": 1, "palette": ["f7fcf5", "74c476", "00441b"]},
    "Vegetation / fuel-type score",
)
add_ee_layer(
    m,
    dryness_score,
    {"min": 0, "max": 1, "palette": ["2166ac", "f7f7f7", "b2182b"]},
    "Typical summer dryness score",
)
add_ee_layer(
    m,
    slope_score,
    {"min": 0, "max": 1, "palette": ["ffffcc", "fd8d3c", "800026"]},
    "Slope score",
)
add_ee_layer(
    m,
    southness,
    {"min": 0, "max": 1, "palette": ["313695", "ffffbf", "a50026"]},
    "Southness score",
)

folium.LayerControl().add_to(m)
m

### Check the factors

Discuss:

1. Which factor has the strongest spatial structure?
2. Which layers are largely independent, and which may be correlated?
3. Where might the WorldCover fuel scores be misleading?
4. What information about fuels is still missing?
5. Why would adding current wind or temperature here change the meaning of the product?

## 8. Build the baseline weighted-overlay model

We use the following baseline weights:

- vegetation / fuel type: **0.40**
- typical summer dryness: **0.30**
- slope: **0.20**
- southness: **0.10**

The weights sum to 1.

They are transparent and editable by design. They are **not statistically calibrated**.

In [ ]:
WEIGHTS = {
    "fuel": 0.40,
    "dryness": 0.30,
    "slope": 0.20,
    "southness": 0.10,
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9

susceptibility = (
    fuel_score.multiply(WEIGHTS["fuel"])
    .add(dryness_score.multiply(WEIGHTS["dryness"]))
    .add(slope_score.multiply(WEIGHTS["slope"]))
    .add(southness.multiply(WEIGHTS["southness"]))
    .updateMask(burnable_mask)
    .rename("susceptibility")
    .clip(AOI)
)

print("Baseline susceptibility model ready.")

## 9. Convert the continuous score into relative classes

We avoid arbitrary universal thresholds.

Instead, the five map classes are defined by the **20th, 40th, 60th and 80th percentiles within the AOI**.

Therefore:

- the classes mean *very low → very high relative to this AOI*;
- they are **not transferable absolute danger classes**.

In [ ]:
sus_pct = (
    susceptibility
    .reduceRegion(
        reducer=ee.Reducer.percentile([20, 40, 60, 80]),
        geometry=AOI,
        scale=30,
        maxPixels=1e9,
        bestEffort=True,
        tileScale=2,
    )
    .getInfo()
)

P20 = float(sus_pct["susceptibility_p20"])
P40 = float(sus_pct["susceptibility_p40"])
P60 = float(sus_pct["susceptibility_p60"])
P80 = float(sus_pct["susceptibility_p80"])

sus_class = (
    ee.Image(1)
    .where(susceptibility.gt(P20), 2)
    .where(susceptibility.gt(P40), 3)
    .where(susceptibility.gt(P60), 4)
    .where(susceptibility.gt(P80), 5)
    .updateMask(susceptibility.mask())
    .rename("susceptibility_class")
)

pd.DataFrame({
    "threshold": ["p20", "p40", "p60", "p80"],
    "score": [P20, P40, P60, P80],
}).round(3)

## 10. Map relative landscape susceptibility

In [ ]:
m = folium.Map(location=CENTER, zoom_start=10)

folium.GeoJson(
    aoi_gdf,
    name="Course AOI",
    style_function=lambda _: {
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.0,
    },
).add_to(m)

effis_map = effis.copy()
effis_map["FIREDATE"] = effis_map["FIREDATE"].dt.strftime("%Y-%m-%d")
if "FINALDATE" in effis_map.columns:
    effis_map["FINALDATE"] = pd.to_datetime(effis_map["FINALDATE"], errors="coerce").dt.strftime("%Y-%m-%d")

folium.GeoJson(
    effis_map,
    name="EFFIS fire history",
    style_function=lambda _: {
        "color": "#222222",
        "weight": 1,
        "fillOpacity": 0.0,
    },
).add_to(m)

add_ee_layer(
    m,
    sus_class,
    {
        "min": 1,
        "max": 5,
        "palette": ["1a9850", "91cf60", "ffffbf", "fc8d59", "d73027"],
    },
    "Relative susceptibility: very low → very high",
)

folium.LayerControl().add_to(m)
m

## 11. Area by susceptibility class

In [ ]:
CLASS_NAMES = {
    1: "Very low",
    2: "Low",
    3: "Moderate",
    4: "High",
    5: "Very high",
}

area_rows = []
for cls, label in CLASS_NAMES.items():
    area_m2 = (
        ee.Image.pixelArea()
        .updateMask(sus_class.eq(cls))
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=AOI,
            scale=30,
            maxPixels=1e9,
            bestEffort=True,
            tileScale=2,
        )
        .get("area")
        .getInfo()
    )

    area_rows.append({
        "class": label,
        "area_km2": area_m2 / 1e6 if area_m2 is not None else np.nan,
    })

class_area = pd.DataFrame(area_rows)
class_area["share_pct"] = (
    class_area["area_km2"] / class_area["area_km2"].sum() * 100
)
class_area.round(2)

## 12. Plausibility check against known EFFIS burned polygons

The weighted overlay was **not fitted to the EFFIS polygons**.

We can therefore ask a modest validation question:

> Do pixels inside the known EFFIS burned footprints tend to occupy higher susceptibility classes than the AOI as a whole?

This is only a plausibility check.

Important limitations:

- EFFIS does not represent every fire;
- burned polygons are not ignition points;
- fire extent depends on weather, suppression and chance;
- some input layers are from different years;
- past fire can itself change vegetation and therefore later susceptibility layers.

In [ ]:
def class_area_table(geometry, label_prefix):
    rows = []

    for cls, class_name in CLASS_NAMES.items():
        area_m2 = (
            ee.Image.pixelArea()
            .updateMask(sus_class.eq(cls))
            .reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=geometry,
                scale=30,
                maxPixels=1e9,
                bestEffort=True,
                tileScale=2,
            )
            .get("area")
            .getInfo()
        )

        rows.append({
            "group": label_prefix,
            "class": class_name,
            "area_km2": area_m2 / 1e6 if area_m2 is not None else 0.0,
        })

    out = pd.DataFrame(rows)
    total = out["area_km2"].sum()
    out["share_pct"] = out["area_km2"] / total * 100 if total > 0 else np.nan
    return out

aoi_classes = class_area[["class", "area_km2", "share_pct"]].copy()
aoi_classes["group"] = "Entire AOI"

effis_classes = class_area_table(EFFIS_GEOM, "Known EFFIS footprints")

comparison = pd.concat([
    aoi_classes[["group", "class", "area_km2", "share_pct"]],
    effis_classes[["group", "class", "area_km2", "share_pct"]],
], ignore_index=True)

comparison.round(2)

In [ ]:
pivot = (
    comparison
    .pivot(index="class", columns="group", values="share_pct")
    .reindex(list(CLASS_NAMES.values()))
)

ax = pivot.plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("Share of area (%)")
ax.set_xlabel("Relative susceptibility class")
ax.set_title("Susceptibility classes: AOI vs known EFFIS burned footprints")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.show()

### Interpret cautiously

A useful result would be a shift of known burned footprints toward the higher relative-susceptibility classes.

But even a strong shift would **not prove causality**.

If there is little or no shift, possible explanations include:

- poor factor choices;
- poor weights;
- missing ignition variables;
- weather dominating the observed fire history;
- temporal mismatch in the input data;
- EFFIS sampling limitations.

## 13. Sensitivity analysis — how much do the weights matter?

Weighted overlays can create an illusion of objectivity because the final map is numerical.

We therefore build two alternative models:

- **fuel-heavy**
- **terrain-heavy**

If the hotspot pattern changes substantially, management conclusions should not depend on one arbitrary weight set.

In [ ]:
def weighted_model(weights):
    assert abs(sum(weights.values()) - 1.0) < 1e-9
    return (
        fuel_score.multiply(weights["fuel"])
        .add(dryness_score.multiply(weights["dryness"]))
        .add(slope_score.multiply(weights["slope"]))
        .add(southness.multiply(weights["southness"]))
        .updateMask(burnable_mask)
        .rename("susceptibility")
        .clip(AOI)
    )

FUEL_HEAVY = {
    "fuel": 0.55,
    "dryness": 0.25,
    "slope": 0.10,
    "southness": 0.10,
}

TERRAIN_HEAVY = {
    "fuel": 0.25,
    "dryness": 0.20,
    "slope": 0.40,
    "southness": 0.15,
}

sus_fuel = weighted_model(FUEL_HEAVY)
sus_terrain = weighted_model(TERRAIN_HEAVY)

diff_fuel = sus_fuel.subtract(susceptibility).rename("fuel_minus_baseline")
diff_terrain = sus_terrain.subtract(susceptibility).rename("terrain_minus_baseline")

print("Alternative models ready.")

In [ ]:
m = folium.Map(location=CENTER, zoom_start=10)

add_ee_layer(
    m,
    diff_fuel,
    {
        "min": -0.20,
        "max": 0.20,
        "palette": ["2166ac", "f7f7f7", "b2182b"],
    },
    "Fuel-heavy minus baseline",
)

add_ee_layer(
    m,
    diff_terrain,
    {
        "min": -0.20,
        "max": 0.20,
        "palette": ["2166ac", "f7f7f7", "b2182b"],
    },
    "Terrain-heavy minus baseline",
)

folium.GeoJson(
    aoi_gdf,
    name="Course AOI",
    style_function=lambda _: {
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.0,
    },
).add_to(m)

folium.LayerControl().add_to(m)
m

### Weight-sensitivity exercise

In groups:

1. Identify one location that remains high under all three models.
2. Identify one location whose score changes strongly.
3. Which factor causes that sensitivity?
4. Would you prioritize a management intervention in a hotspot that disappears after a small change in weights?
5. What evidence would help calibrate the weights more defensibly?

## 14. From susceptibility to management

A susceptibility map can support questions such as:

- Where should field teams inspect fuel continuity or vegetation condition?
- Which areas might deserve extra attention when **fire weather becomes severe**?
- Where could fuel-management scenarios be tested?
- Which locations are robust hotspots across multiple plausible weighting schemes?

It cannot by itself answer:

- Where will the next fire ignite?
- What is today's fire danger?
- Which communities or assets are at greatest risk?
- What suppression strategy should be used during an active incident?

Those require dynamic weather, ignition information, exposure, vulnerability and operational context.

## 15. Optional stretch tasks

Choose one:

### A — Add a human-accessibility proxy
Explore distance to roads, settlements or agricultural edges. Explain why this changes the model from mainly **landscape susceptibility** toward **occurrence susceptibility**.

### B — Data-driven model
Use historical fire observations and carefully selected background samples to fit logistic regression or Random Forest.

Do **not** use a random pixel split only. Spatial autocorrelation can make accuracy look much better than real transferability. Use spatially separated validation.

### C — Cross-validation by year
Train on earlier fire years and test on later years.

### D — Alternative dryness metric
Replace typical summer NDMI with another defensible vegetation-moisture or drought proxy and compare the maps.

### E — Dynamic hazard
Combine the susceptibility map conceptually with the fire-weather diagnostics from Practical 05. Be explicit that the resulting product is no longer purely susceptibility.

### F — Management threshold
Instead of equal-area quintiles, define a management threshold based on a concrete decision and justify it.

## 16. Output for the Galičica capstone

Keep:

- the four factor maps or one compact factor summary;
- the baseline susceptibility map;
- the AOI vs EFFIS class comparison;
- one sensitivity-analysis result;
- **three defensible findings**;
- **two limitations**;
- **one management-relevant recommendation**.

A strong final statement should sound like:

> “Under this transparent set of landscape assumptions, these areas are relatively more susceptible, and the pattern is reasonably robust to plausible weight changes.”

It should **not** sound like:

> “These pixels have an 80% probability of burning.”

You have now completed the historical fire → burned area/severity → recovery → weather/fuel condition → susceptibility sequence.